# Chatbot: Simple (no memory yet)

`START -> chat_node -> END`

The state is a list of messages. The chat node sends them to the LLM and adds the reply.

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')

In [2]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  # add_messages appends new messages

In [3]:
def chat_node(state: ChatState):
    response = model.invoke(state['messages'])
    return {'messages': [response]}

In [4]:
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile()

In [5]:
result = chatbot.invoke({'messages': [HumanMessage(content='Hi, my name is Priyansh')]})
for message in result['messages']:
    print(f'{message.type}: {message.content}')

human: Hi, my name is Priyansh
ai: Hello, Priyansh! Nice to meet you. How can I help you today?


In [6]:
# Each invoke starts from an empty state, so the bot has already forgotten the name
result = chatbot.invoke({'messages': [HumanMessage(content='What is my name?')]})
print(result['messages'][-1].content)

I don’t actually know your name—could you let me know what you’d like me to call you?


In [ ]:
# Chat in a loop. Type 'exit' to stop.
while True:
    user_message = input('You: ')
    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break
    result = chatbot.invoke({'messages': [HumanMessage(content=user_message)]})
    print('AI:', result['messages'][-1].content)